# Notebook 79: V8.9 151-page reconstruction diagnostic panels

Creates one 2×2 review panel for every page in the latest completed Notebook 78 run:

1. **Observed**: annotated input page.
2. **V8.9 final**: frozen corrected output.
3. **No reconstruction**: the canonical V7.8 route treatment with `REMOVE → paper`, while `AMBIGUOUS → observed` rather than Telea. `PRESERVE → observed`. This is a diagnostic, not a production candidate.
4. **Route overlay**: route ownership on the observed page, with **blue = PRESERVE**, **green = REMOVE**, **orange = AMBIGUOUS**, and **cyan = V8.9 Ring 2**.

The notebook does not retrain, recalibrate, alter the frozen V8.9 output, or use clean truth. It reads the existing 151-page Notebook 78 run and writes separate diagnostic panels.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy matplotlib


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, shutil
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('/content/drive/MyDrive/MusicAnnotationProject')
NB78_ROOT = ROOT / 'v8_9_release'
EXPECTED_PAGES = 151

# Optional: set this to one exact completed Notebook 78 run.
NB78_RUN_OVERRIDE = None

RUN = ROOT / 'v8_9_reconstruction_diagnostic' / (
    datetime.now().strftime('%Y%m%d_%H%M%S') + '_notebook79'
)
PANELS = RUN / 'panels'
TABLES = RUN / 'tables'
AUDIT = RUN / 'audit'
UPLOAD = RUN / 'upload'
for p in [RUN, PANELS, TABLES, AUDIT, UPLOAD]:
    p.mkdir(parents=True, exist_ok=False)


def require(value, message):
    if not bool(value):
        raise RuntimeError(message)


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1048576), b''):
            h.update(block)
    return h.hexdigest()


def complete_nb78_runs(root):
    required = [
        'release_verdict.json',
        'audit/151_page_regeneration.csv',
        'pages',
    ]
    return sorted(
        [p for p in Path(root).iterdir()
         if p.is_dir() and all((p / r).exists() for r in required)],
        key=lambda p: p.stat().st_mtime,
    ) if Path(root).exists() else []

if NB78_RUN_OVERRIDE is not None:
    NB78 = Path(NB78_RUN_OVERRIDE)
else:
    hits = complete_nb78_runs(NB78_ROOT)
    require(hits, f'No complete Notebook 78 run found below {NB78_ROOT}')
    NB78 = hits[-1]

verdict = json.loads((NB78 / 'release_verdict.json').read_text())
require(verdict.get('verdict') == 'PASS', 'Notebook 78 release verdict is not PASS.')
roster = pd.read_csv(NB78 / 'audit' / '151_page_regeneration.csv')
require(len(roster) == EXPECTED_PAGES, f'Expected 151 pages, found {len(roster)}.')
require('page' in roster.columns, 'Notebook 78 regeneration table lacks page.')

print('Notebook 78:', NB78)
print('Notebook 79:', RUN)
print('Pages:', len(roster))


## Canonical route reconstruction

The route diagnostic follows the frozen V7.8 implementation: the reviewed seed is dilated by 2, Telea is computed on the prepared mask, and `geometry_t80` divides the mask into preserve/remove/ambiguous. The diagnostic then suppresses only the Telea treatment by leaving ambiguous pixels observed. The production V8.9 page remains unchanged in the top-right panel. The canonical project contract routes `REMOVE` to the local paper estimate, `AMBIGUOUS` to Telea, and `PRESERVE` to the observed byte. citeturn119search26L145-L159


In [ ]:
MASK_DILATION = 2
CROP_MARGIN = 48
GEOMETRY_THRESHOLD = 80
BAND_WIDTH = 4


def binary(mask):
    return np.where(mask > 0, 255, 0).astype(np.uint8)


def prepare_mask(seed, dilation=MASK_DILATION):
    mask = binary(seed)
    if dilation > 0:
        kernel = cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE,
            (2 * dilation + 1, 2 * dilation + 1),
        )
        mask = cv2.dilate(mask, kernel)
    return mask


def background_surface(observed, mask, sigma=45.0):
    visible = (mask == 0) & (observed >= 200)
    values = observed.astype(np.float32) * visible
    weights = visible.astype(np.float32)
    numerator = cv2.GaussianBlur(values, (0, 0), sigma)
    denominator = cv2.GaussianBlur(weights, (0, 0), sigma)
    surface = numerator / np.maximum(denominator, 1e-4)
    fallback = cv2.GaussianBlur(observed, (0, 0), sigma)
    return np.clip(
        np.where(denominator > 0.02, surface, fallback),
        0,
        255,
    ).astype(np.uint8)


def support_maps(observed, mask):
    dark = (observed < 190) & (mask == 0)
    horizontal = cv2.morphologyEx(
        dark.astype(np.uint8) * 255,
        cv2.MORPH_OPEN,
        np.ones((1, 35), np.uint8),
    ) > 0
    vertical = cv2.morphologyEx(
        dark.astype(np.uint8) * 255,
        cv2.MORPH_OPEN,
        np.ones((15, 1), np.uint8),
    ) > 0
    n, labels, stats, _ = cv2.connectedComponentsWithStats(
        dark.astype(np.uint8), 8
    )
    compact = np.zeros_like(dark)
    for component_id in range(1, n):
        x, y, w, h, area = stats[component_id]
        if 3 <= area <= 650 and w <= 40 and h <= 40:
            compact[labels == component_id] = True
    kernel = np.ones((2 * BAND_WIDTH + 1, 2 * BAND_WIDTH + 1), np.uint8)
    return (
        cv2.dilate(horizontal.astype(np.uint8), kernel) > 0,
        cv2.dilate(vertical.astype(np.uint8), kernel) > 0,
        cv2.dilate(compact.astype(np.uint8), kernel) > 0,
    )


def geometry_routes(observed, prepared_mask):
    paper = background_surface(observed, prepared_mask)
    normalised = cv2.divide(observed, np.maximum(paper, 1), scale=255)
    horizontal, vertical, compact = support_maps(observed, prepared_mask)
    authority = prepared_mask > 0
    preserve = authority & (horizontal | vertical | compact) & (observed < 200)
    remove = (
        authority
        & ~preserve
        & (observed >= GEOMETRY_THRESHOLD)
        & (normalised >= GEOMETRY_THRESHOLD)
    )
    ambiguous = authority & ~preserve & ~remove
    return {
        'paper': paper,
        'preserve': preserve,
        'remove': remove,
        'ambiguous': ambiguous,
    }


def ring2_from_seed(seed):
    seed_bool = seed > 0
    d1 = cv2.dilate(
        seed_bool.astype(np.uint8),
        cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3)),
    ) > 0
    d2 = cv2.dilate(
        seed_bool.astype(np.uint8),
        cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)),
    ) > 0
    return d2 & ~d1


def no_reconstruction_page(observed, routes):
    # Diagnostic isolation of Telea's role:
    # REMOVE -> paper; AMBIGUOUS -> observed; PRESERVE -> observed.
    result = observed.copy()
    result[routes['remove']] = routes['paper'][routes['remove']]
    return result


def route_overlay(observed, routes, ring2, alpha=0.58):
    base = cv2.cvtColor(observed, cv2.COLOR_GRAY2RGB)
    colour = base.copy()
    active = np.zeros(observed.shape, bool)
    layers = [
        (routes['preserve'], (35, 105, 245)),   # blue
        (routes['remove'], (45, 185, 75)),      # green
        (routes['ambiguous'], (245, 145, 25)),  # orange
        (ring2, (25, 205, 215)),                # cyan
    ]
    for mask, rgb in layers:
        colour[mask] = rgb
        active |= mask
    output = base.copy()
    output[active] = (
        base[active].astype(np.float32) * (1 - alpha)
        + colour[active].astype(np.float32) * alpha
    ).astype(np.uint8)
    return output


## Generate the 151 panels

This stage is read-only with respect to Notebook 78. It requires the per-page `observed.png`, `authoritative_seed_mask.png`, and `v8_9.png` files generated by Notebook 78.


In [ ]:
rows = []

for page in roster.page.astype(str):
    page_dir = NB78 / 'pages' / page
    observed_path = page_dir / 'observed.png'
    seed_path = page_dir / 'authoritative_seed_mask.png'
    v89_path = page_dir / 'v8_9.png'

    require(observed_path.exists(), f'{page}: missing {observed_path}')
    require(seed_path.exists(), f'{page}: missing {seed_path}')
    require(v89_path.exists(), f'{page}: missing {v89_path}')

    observed = cv2.imread(str(observed_path), cv2.IMREAD_GRAYSCALE)
    seed = cv2.imread(str(seed_path), cv2.IMREAD_GRAYSCALE)
    v89 = cv2.imread(str(v89_path), cv2.IMREAD_GRAYSCALE)

    require(
        observed is not None and seed is not None and v89 is not None,
        f'{page}: unreadable input image.',
    )
    require(
        observed.shape == seed.shape == v89.shape,
        f'{page}: shape mismatch.',
    )

    prepared = prepare_mask(seed, 2)
    routes = geometry_routes(observed, prepared)
    ring2 = ring2_from_seed(seed)
    no_recon = no_reconstruction_page(observed, routes)
    overlay = route_overlay(observed, routes, ring2)

    # Fail closed on route partition and collar identity.
    route_union = routes['preserve'] | routes['remove'] | routes['ambiguous']
    route_overlap = (
        routes['preserve'].astype(np.uint8)
        + routes['remove'].astype(np.uint8)
        + routes['ambiguous'].astype(np.uint8)
    )
    require(np.array_equal(route_union, prepared > 0), f'{page}: routes do not cover authority.')
    require(int((route_overlap > 1).sum()) == 0, f'{page}: route overlap detected.')
    require(np.all(ring2 <= (prepared > 0)), f'{page}: Ring 2 escapes prepared authority.')

    fig, axes = plt.subplots(2, 2, figsize=(14, 18), constrained_layout=True)
    panels = [observed, v89, no_recon, overlay]
    titles = [
        'Observed annotated page',
        'Frozen V8.9 final',
        'No reconstruction diagnostic\nREMOVE → paper; AMBIGUOUS/PRESERVE → observed',
        'Route overlay\nBlue preserve | Green remove | Orange ambiguous | Cyan Ring 2',
    ]

    for ax, image, title in zip(axes.flat, panels, titles):
        if image.ndim == 2:
            ax.imshow(image, cmap='gray', vmin=0, vmax=255)
        else:
            ax.imshow(image)
        ax.set_title(title, fontsize=11)
        ax.axis('off')

    fig.suptitle(page, fontsize=14)
    safe_name = ''.join(c if c.isalnum() or c in '._-' else '_' for c in page)
    panel_path = PANELS / f'{safe_name}__v89_reconstruction_2x2.png'
    fig.savefig(panel_path, dpi=135, bbox_inches='tight')
    plt.close(fig)

    rows.append({
        'page': page,
        'panel': str(panel_path),
        'panel_sha256': sha256_file(panel_path),
        'prepared_mask_pixels': int((prepared > 0).sum()),
        'preserve_pixels': int(routes['preserve'].sum()),
        'remove_pixels': int(routes['remove'].sum()),
        'ambiguous_pixels': int(routes['ambiguous'].sum()),
        'ring2_pixels': int(ring2.sum()),
        'v89_changed_from_observed': int((v89 != observed).sum()),
        'no_reconstruction_changed_from_observed': int((no_recon != observed).sum()),
        'v89_vs_no_reconstruction_different_pixels': int((v89 != no_recon).sum()),
    })

    print(page)

panel_index = pd.DataFrame(rows)
panel_index.to_csv(TABLES / 'panel_index.csv', index=False)
require(len(panel_index) == EXPECTED_PAGES, 'Panel count is not 151.')
print('Generated panels:', len(panel_index))


## Contact sheets, manifest and ZIP

The notebook also creates multiple contact sheets rather than one unreadably large bitmap. Pages are ranked by the difference between the frozen V8.9 page and the no-reconstruction diagnostic.


In [ ]:
ranked = panel_index.sort_values(
    'v89_vs_no_reconstruction_different_pixels',
    ascending=False,
).reset_index(drop=True)
ranked.to_csv(TABLES / 'panel_index_ranked.csv', index=False)

CONTACT_PAGES = 12
contact_paths = []
for start in range(0, len(ranked), CONTACT_PAGES):
    subset = ranked.iloc[start:start + CONTACT_PAGES]
    fig, axes = plt.subplots(3, 4, figsize=(20, 24), constrained_layout=True)
    for ax in axes.flat:
        ax.axis('off')
    for ax, (_, row) in zip(axes.flat, subset.iterrows()):
        image = cv2.imread(str(row.panel), cv2.IMREAD_COLOR)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        ax.imshow(image)
        ax.set_title(
            f"{row.page}\nV8.9 vs no-recon: "
            f"{int(row.v89_vs_no_reconstruction_different_pixels):,} px",
            fontsize=9,
        )
        ax.axis('off')
    contact_path = RUN / f'contact_sheet_{start // CONTACT_PAGES + 1:02d}.png'
    fig.savefig(contact_path, dpi=110, bbox_inches='tight')
    plt.close(fig)
    contact_paths.append(contact_path)

manifest = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'notebook': '79_V8_9_151_Page_Reconstruction_Diagnostic_Panels',
    'source_notebook78_run': str(NB78),
    'source_notebook78_verdict': verdict.get('verdict'),
    'pages': int(len(panel_index)),
    'panel_definition': {
        'top_left': 'observed annotated input',
        'top_right': 'frozen V8.9 final output',
        'bottom_left': 'REMOVE to paper; AMBIGUOUS and PRESERVE retained observed',
        'bottom_right': 'observed page route overlay plus V8.9 Ring 2',
    },
    'diagnostic_only': True,
    'clean_truth_used': False,
    'v8_9_outputs_modified': False,
    'panels_directory': str(PANELS),
    'contact_sheets': [str(p) for p in contact_paths],
}
(RUN / 'panel_manifest.json').write_text(json.dumps(manifest, indent=2))

# Curated upload package: manifest, tables, contact sheets and all 151 panels.
for path in [
    RUN / 'panel_manifest.json',
    TABLES / 'panel_index.csv',
    TABLES / 'panel_index_ranked.csv',
    *contact_paths,
]:
    shutil.copy2(path, UPLOAD / path.name)

shutil.copytree(PANELS, UPLOAD / 'panels')
zip_base = RUN / 'notebook79_v89_reconstruction_panels'
shutil.make_archive(str(zip_base), 'zip', UPLOAD)

print(json.dumps(manifest, indent=2))
print('Bundle:', str(zip_base) + '.zip')
